# Notebook 03 — Model Training
**Mental Health Assessment Using Machine Learning**

---

## Purpose

This notebook trains all baseline machine learning and deep learning models across the nine feature sets produced by Notebook 02. It produces 72 trained ML models, 18 trained DL models, 108 evaluation figures, and result CSVs.

---

## Primary Metric: Macro F1

The test set is class-imbalanced (Stable: 6.2%, Challenged: 29.9%, Critical: 64.0%). A model predicting only the majority class would achieve 64% accuracy without learning anything. **Macro F1 is therefore the primary evaluation and ranking metric throughout this notebook** — it weights each class equally regardless of size. Accuracy is reported alongside it for completeness but is not used for ranking.

This applies to both ML and DL models:
- **ML:** result CSVs and summary table sorted by `F1_Macro`
- **DL:** early stopping maximises validation macro F1 (not minimises validation loss); training plots show macro F1 vs epoch

---

## Machine Learning Models (8 Algorithms × 9 Feature Sets = 72 Models)

All classifiers use default hyperparameters — optimisation is in Notebook 04. All sklearn estimators use `n_jobs=-1`.

| Key | Algorithm | Configuration |
|-----|-----------|---------------|
| `lr` | Logistic Regression | `max_iter=1000`, L2 regularisation |
| `dt` | Decision Tree | Gini criterion, unlimited depth |
| `rf` | Random Forest | 100 trees, Gini criterion |
| `knn` | K-Nearest Neighbours | k=5, Euclidean distance |
| `svm` | Support Vector Machine | RBF kernel, `probability=True` |
| `gb` | Gradient Boosting | 100 estimators, lr=0.1 |
| `xgb` | XGBoost | `eval_metric=mlogloss` |
| `lgbm` | LightGBM | Leaf-wise growth |

**Metrics recorded:** Accuracy · Precision (weighted) · Recall (weighted) · F1 (weighted) · F1 (macro)

---

## Deep Learning Models

Both architectures trained with Adam (lr=0.001), CrossEntropyLoss, batch=32, max 50 epochs, early stopping patience=7 on a 20% stratified validation split of the SMOTE training set.

**Early stopping criterion:** validation macro F1 is computed after every epoch. Training stops when it fails to improve for 7 consecutive epochs. The weights from the epoch with the highest validation macro F1 are restored.

**ANN:** `Linear(15→128) → ReLU → Dropout(0.3) → Linear(128→64) → ReLU → Linear(64→3)`

**CNN:** `Input(B,1,15) → Conv1d(1→32,k=3,pad=1) → ReLU → MaxPool1d(2) → Conv1d(32→64,k=3,pad=1) → ReLU → Flatten(448) → Linear(448→64) → ReLU → Dropout(0.3) → Linear(64→3)`

---

## Cell Map

| Cell | Summary |
|------|---------|
| 1 | Imports · CPU thread config · constants |
| 2 | Load all 9 feature sets from disk |
| 3 | Train 8 ML algorithms × 9 feature sets (72 models) |
| 4 | ML results summary CSV — sorted by F1_Macro |
| 5 | Define ANN · training helpers (early stop on val macro F1) · train ANN × 9 |
| 6 | Define CNN1D · train CNN × 9 |
| 7 | DL results summary CSV — sorted by F1_Macro |

## Cell 1 — Imports and CPU Configuration

Imports all libraries. PyTorch is configured to use all available CPU threads. `f1_score` from sklearn is imported for use inside the DL training loop validation step.

In [1]:
from pathlib import Path
import os, warnings, copy
warnings.filterwarnings('ignore')

_cwd = Path.cwd()
if _cwd.name == 'notebooks':
    os.chdir(_cwd.parent)
print(f'Working directory: {Path.cwd()}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.svm             import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics         import (accuracy_score, precision_score, recall_score,
                                      f1_score, confusion_matrix, ConfusionMatrixDisplay)
import xgboost  as xgb
import lightgbm as lgb

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.set_num_threads(os.cpu_count())
plt.rcParams.update({'savefig.dpi': 300,
                     'axes.spines.top': False,
                     'axes.spines.right': False})
sns.set_palette('Set2')

CLASS_NAMES = ['Stable', 'Challenged', 'Critical']
METHODS     = ['rfe', 'skb', 'fscs', 'etc', 'pc', 'mi', 'mir', 'mu', 'vt']
SEED        = 42

print(f'PyTorch threads : {torch.get_num_threads()}')
print(f'CPU cores       : {os.cpu_count()}')
print('\n✓ All imports successful')

Working directory: d:\Programming\Projects\Mental Health Assessment
PyTorch threads : 8
CPU cores       : 8

✓ All imports successful


## Cell 2 — Load All 9 Feature Sets from Disk

Reconstructs `feature_sets` by reading the `train.csv` and `test.csv` files produced by Notebook 02. The `label` column (integers 0/1/2) is separated from the 15 feature columns by name. A summary table confirms all 9 sets are present with the correct dimensions and class distributions before any training begins.

In [2]:
feature_sets = {}

for method in METHODS:
    train_df  = pd.read_csv(os.path.join('features', 'Tabular', method, 'train.csv'))
    test_df   = pd.read_csv(os.path.join('features', 'Tabular', method, 'test.csv'))
    feat_cols = [c for c in train_df.columns if c != 'label']
    feature_sets[method] = {
        'train_X': train_df[feat_cols].values,
        'train_y': train_df['label'].values.astype(int),
        'test_X' : test_df[feat_cols].values,
        'test_y' : test_df['label'].values.astype(int),
    }

print(f'{'Method':<6}  {'Train shape':<16}  {'Test shape':<14}  Class dist (train)')
print('-' * 75)
for method, fd in feature_sets.items():
    dist = dict(zip(CLASS_NAMES, np.bincount(fd['train_y'])))
    print(f'{method.upper():<6}  {str(fd["train_X"].shape):<16}  '
          f'{str(fd["test_X"].shape):<14}  {dist}')

print(f'\n✓ {len(feature_sets)} feature sets loaded')

Method  Train shape       Test shape      Class dist (train)
---------------------------------------------------------------------------
RFE     (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
SKB     (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
FSCS    (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
ETC     (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
PC      (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
MI      (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'Critical': np.int64(1033)}
MIR     (3099, 15)        (405, 15)       {'Stable': np.int64(1033), 'Challenged': np.int64(1033), 'C

## Cell 3 — Train 8 ML Algorithms × 9 Feature Sets (72 Models)

Iterates over all 9 feature sets and all 8 algorithms. For each combination: the classifier is deep-copied, fit on the SMOTE training set, and evaluated on the held-out test set. Five metrics are computed and recorded. The trained model is saved as `{algo}.pkl` and its confusion matrix saved as a 300 DPI PNG. A per-method result CSV is written after all 8 algorithms complete for that feature set.

In [3]:
ALGORITHMS = {
    'lr'  : LogisticRegression(max_iter=1000, random_state=SEED, n_jobs=-1),
    'dt'  : DecisionTreeClassifier(random_state=SEED),
    'rf'  : RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'knn' : KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'svm' : SVC(probability=True, random_state=SEED),
    'gb'  : GradientBoostingClassifier(random_state=SEED),
    'xgb' : xgb.XGBClassifier(random_state=SEED, n_jobs=-1,
                               eval_metric='mlogloss', verbosity=0),
    'lgbm': lgb.LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1),
}

all_ml_results = []

for method, fd in feature_sets.items():
    print(f'\n── {method.upper()} ──')
    X_tr, y_tr = fd['train_X'], fd['train_y']
    X_te, y_te = fd['test_X'],  fd['test_y']
    method_results = []

    for algo_name, algo in ALGORITHMS.items():
        clf    = copy.deepcopy(algo)
        clf.fit(X_tr, y_tr)
        y_pred = clf.predict(X_te)

        acc  = accuracy_score(y_te, y_pred)
        prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
        rec  = recall_score(y_te,   y_pred,  average='weighted', zero_division=0)
        f1w  = f1_score(y_te,       y_pred,  average='weighted', zero_division=0)
        f1m  = f1_score(y_te,       y_pred,  average='macro',    zero_division=0)
        print(f'  {algo_name.upper():<5}: F1_macro={f1m:.4f}  Acc={acc:.4f}')

        model_dir = os.path.join('models', 'Machine Learning', method)
        os.makedirs(model_dir, exist_ok=True)
        joblib.dump(clf, os.path.join(model_dir, f'{algo_name}.pkl'))

        fig_dir = os.path.join('figures', 'Machine Learning', method)
        os.makedirs(fig_dir, exist_ok=True)
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(
            confusion_matrix(y_te, y_pred), display_labels=CLASS_NAMES
        ).plot(ax=ax, cmap='Blues', colorbar=False)
        ax.set_title(f'{algo_name.upper()} | {method.upper()}', fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, f'{algo_name}_confusion.png'),
                    dpi=300, bbox_inches='tight')
        plt.close()

        row = {
            'Feature_Method'    : method,
            'Model'             : algo_name.upper(),
            'Accuracy'          : round(acc,  4),
            'Precision_Weighted': round(prec, 4),
            'Recall_Weighted'   : round(rec,  4),
            'F1_Weighted'       : round(f1w,  4),
            'F1_Macro'          : round(f1m,  4),
        }
        method_results.append(row)
        all_ml_results.append(row)

    res_dir = os.path.join('results', 'Machine Learning', method)
    os.makedirs(res_dir, exist_ok=True)
    pd.DataFrame(method_results).to_csv(
        os.path.join(res_dir, 'results_traditional_ml.csv'), index=False
    )

print('\n✓ All 72 ML models trained and saved')


── RFE ──
  LR   : F1_macro=0.7671  Acc=0.8444
  DT   : F1_macro=0.7848  Acc=0.8444
  RF   : F1_macro=0.8410  Acc=0.8963
  KNN  : F1_macro=0.7988  Acc=0.8568
  SVM  : F1_macro=0.8288  Acc=0.8864
  GB   : F1_macro=0.8537  Acc=0.8864
  XGB  : F1_macro=0.8358  Acc=0.8963
  LGBM : F1_macro=0.8438  Acc=0.8963

── SKB ──
  LR   : F1_macro=0.7659  Acc=0.8420
  DT   : F1_macro=0.8044  Acc=0.8667
  RF   : F1_macro=0.8490  Acc=0.9111
  KNN  : F1_macro=0.8080  Acc=0.8765
  SVM  : F1_macro=0.8336  Acc=0.8914
  GB   : F1_macro=0.8563  Acc=0.9012
  XGB  : F1_macro=0.8632  Acc=0.9136
  LGBM : F1_macro=0.8502  Acc=0.9086

── FSCS ──
  LR   : F1_macro=0.7526  Acc=0.8395
  DT   : F1_macro=0.8016  Acc=0.8716
  RF   : F1_macro=0.8438  Acc=0.8963
  KNN  : F1_macro=0.7694  Acc=0.8346
  SVM  : F1_macro=0.7871  Acc=0.8642
  GB   : F1_macro=0.8431  Acc=0.8889
  XGB  : F1_macro=0.8663  Acc=0.9160
  LGBM : F1_macro=0.8396  Acc=0.8963

── ETC ──
  LR   : F1_macro=0.7678  Acc=0.8469
  DT   : F1_macro=0.7870  Acc=

## Cell 4 — ML Results Summary CSV

Aggregates all 72 rows into a single DataFrame and saves to `summary/Results/Machine Learning/ml_results_summary.csv`. The table is sorted by **F1_Macro** (descending) — macro F1 is the primary ranking metric given the class imbalance in the test set. The top 10 are printed to surface the best-performing feature method and algorithm combination.

In [4]:
ml_summary = pd.DataFrame(all_ml_results)

os.makedirs(os.path.join('summary', 'Results', 'Machine Learning'), exist_ok=True)
ML_SUMMARY = os.path.join('summary', 'Results', 'Machine Learning', 'ml_results_summary.csv')
ml_summary.to_csv(ML_SUMMARY, index=False)

print(f'✓ Saved : {ML_SUMMARY}')
print(f'  Shape : {ml_summary.shape}  (expected 72 rows × 7 cols)')
print()
print('Top 10 by F1_Macro (primary metric):')
print(
    ml_summary
    .sort_values('F1_Macro', ascending=False)
    [['Feature_Method', 'Model', 'F1_Macro', 'Accuracy']]
    .head(10)
    .to_string(index=False)
)

✓ Saved : summary\Results\Machine Learning\ml_results_summary.csv
  Shape : (72, 7)  (expected 72 rows × 7 cols)

Top 10 by F1_Macro (primary metric):
Feature_Method Model  F1_Macro  Accuracy
          fscs   XGB    0.8663    0.9160
           skb   XGB    0.8632    0.9136
            pc   XGB    0.8632    0.9136
            mi   XGB    0.8586    0.9136
           mir   XGB    0.8586    0.9136
           skb    GB    0.8563    0.9012
            pc    GB    0.8563    0.9012
           etc    GB    0.8558    0.8963
           rfe    GB    0.8537    0.8864
            mu   XGB    0.8534    0.9037


## Cell 5 — ANN Architecture, Training Helpers, and ANN × 9 Feature Sets

Defines the `ANN` class and four helper functions reused by both Cell 5 (ANN) and Cell 6 (CNN).

**`train_dl` — early stopping on validation macro F1:**  
After each epoch the model is switched to eval mode and validation macro F1 is computed from the argmax predictions. If it improves over the previous best, the model weights are saved via `state_dict()`. If it fails to improve for `patience=7` consecutive epochs, training stops and the best weights are restored. CrossEntropyLoss is still used as the optimisation objective during the forward-backward pass — macro F1 is used only for monitoring and early stopping.

**`save_dl_plots` — two-panel training figure:**  
Left panel: training loss + validation loss vs epoch (monitors convergence).  
Right panel: validation macro F1 vs epoch (monitors generalisation on the primary metric).

**`save_dl_cm`** — confusion matrix for test-set predictions.  
**`eval_metrics`** — returns all five evaluation metrics in one call.

In [5]:
class ANN(nn.Module):
    """Linear(15→128)→ReLU→Dropout(0.3)→Linear(128→64)→ReLU→Linear(64→3)"""
    def __init__(self, input_dim=15, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, 3)
        )
    def forward(self, x): return self.net(x)


def train_dl(model, X_tr, y_tr, X_val, y_val,
             epochs=50, batch_size=32, patience=7, lr=0.001):
    """Train with Adam + CrossEntropyLoss. Early stopping maximises val macro F1."""
    optimizer  = torch.optim.Adam(model.parameters(), lr=lr)
    criterion  = nn.CrossEntropyLoss()
    Xtr_t  = torch.FloatTensor(X_tr);  ytr_t  = torch.LongTensor(y_tr)
    Xval_t = torch.FloatTensor(X_val); yval_t = torch.LongTensor(y_val)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True)
    history    = {'train_loss': [], 'val_loss': [], 'val_f1_macro': []}
    best_f1, no_imp, best_state = 0.0, 0, None

    for epoch in range(epochs):
        # Training step
        model.train()
        tl, tt = 0.0, 0
        for Xb, yb in loader:
            out  = model(Xb); loss = criterion(out, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item() * len(yb); tt += len(yb)

        # Validation step — compute loss and macro F1
        model.eval()
        with torch.no_grad():
            vo        = model(Xval_t)
            vl        = criterion(vo, yval_t).item()
            val_preds = vo.argmax(1).numpy()
        val_f1 = f1_score(y_val, val_preds, average='macro', zero_division=0)

        history['train_loss'].append(tl / tt)
        history['val_loss'].append(vl)
        history['val_f1_macro'].append(val_f1)

        # Early stopping: maximise validation macro F1
        if val_f1 > best_f1:
            best_f1, no_imp = val_f1, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f'    Early stop at epoch {epoch + 1}  '
                      f'(best val F1_macro={best_f1:.4f})')
                break

    if best_state:
        model.load_state_dict(best_state)
    return model, history


def save_dl_plots(history, model_name, method, fig_dir):
    """Two-panel figure: loss curves (left) and validation macro F1 (right)."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'],   label='Val Loss')
    axes[0].set_title(f'{model_name} Loss — {method.upper()}')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('CrossEntropy Loss')
    axes[0].legend()

    axes[1].plot(history['val_f1_macro'], label='Val Macro F1', color='steelblue')
    axes[1].axhline(y=max(history['val_f1_macro']), color='red',
                    linestyle='--', linewidth=0.8,
                    label=f'Best: {max(history["val_f1_macro"]):.4f}')
    axes[1].set_title(f'{model_name} Val Macro F1 — {method.upper()}')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')
    axes[1].set_ylim(0, 1); axes[1].legend()

    plt.tight_layout()
    fname = 'ann_training.png' if model_name == 'ANN' else 'cnn_training.png'
    plt.savefig(os.path.join(fig_dir, fname), dpi=300, bbox_inches='tight')
    plt.close()


def save_dl_cm(y_te, y_pred, model_name, method, fig_dir):
    """Confusion matrix for test-set predictions."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(y_te, y_pred), display_labels=CLASS_NAMES
    ).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{model_name} | {method.upper()}', fontweight='bold')
    plt.tight_layout()
    fname = 'ann_confusion.png' if model_name == 'ANN' else 'cnn_confusion.png'
    plt.savefig(os.path.join(fig_dir, fname), dpi=300, bbox_inches='tight')
    plt.close()


def eval_metrics(y_te, y_pred):
    """Return dict of five evaluation metrics. F1_Macro is the primary metric."""
    return {
        'Accuracy'          : round(accuracy_score(y_te, y_pred), 4),
        'Precision_Weighted': round(precision_score(y_te, y_pred, average='weighted', zero_division=0), 4),
        'Recall_Weighted'   : round(recall_score(y_te,   y_pred, average='weighted', zero_division=0), 4),
        'F1_Weighted'       : round(f1_score(y_te,       y_pred, average='weighted', zero_division=0), 4),
        'F1_Macro'          : round(f1_score(y_te,       y_pred, average='macro',    zero_division=0), 4),
    }


# ANN training — 9 feature sets
all_dl_results = []

for method, fd in feature_sets.items():
    print(f'\n── ANN | {method.upper()} ──')
    X_tr_all, y_tr_all = fd['train_X'], fd['train_y']
    X_te,     y_te     = fd['test_X'],  fd['test_y']

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_all, y_tr_all, test_size=0.2, random_state=SEED, stratify=y_tr_all
    )

    model = ANN(input_dim=X_tr.shape[1])
    model, history = train_dl(model, X_tr, y_tr, X_val, y_val)

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.FloatTensor(X_te)).argmax(1).numpy()

    metrics = eval_metrics(y_te, y_pred)
    print(f'  Test F1_macro={metrics["F1_Macro"]}  Acc={metrics["Accuracy"]}')

    dl_dir  = os.path.join('models',  'Deep Learning', method)
    fig_dir = os.path.join('figures', 'Deep Learning', method)
    os.makedirs(dl_dir,  exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)

    torch.save(model.state_dict(), os.path.join(dl_dir, 'ann_model.pt'))
    save_dl_plots(history, 'ANN', method, fig_dir)
    save_dl_cm(y_te, y_pred, 'ANN', method, fig_dir)

    all_dl_results.append({'Feature_Method': method, 'Model': 'ANN', **metrics})

print('\n✓ All 9 ANN models trained and saved')


── ANN | RFE ──
    Early stop at epoch 28  (best val F1_macro=0.9597)
  Test F1_macro=0.8453  Acc=0.9062

── ANN | SKB ──
    Early stop at epoch 25  (best val F1_macro=0.9580)
  Test F1_macro=0.8296  Acc=0.9037

── ANN | FSCS ──
    Early stop at epoch 19  (best val F1_macro=0.9484)
  Test F1_macro=0.8081  Acc=0.8815

── ANN | ETC ──
    Early stop at epoch 24  (best val F1_macro=0.9582)
  Test F1_macro=0.8577  Acc=0.921

── ANN | PC ──
    Early stop at epoch 43  (best val F1_macro=0.9564)
  Test F1_macro=0.8343  Acc=0.9062

── ANN | MI ──
    Early stop at epoch 35  (best val F1_macro=0.9630)
  Test F1_macro=0.8435  Acc=0.9136

── ANN | MIR ──
    Early stop at epoch 22  (best val F1_macro=0.9548)
  Test F1_macro=0.8355  Acc=0.9136

── ANN | MU ──
    Early stop at epoch 28  (best val F1_macro=0.9646)
  Test F1_macro=0.8484  Acc=0.9136

── ANN | VT ──
    Early stop at epoch 25  (best val F1_macro=0.9178)
  Test F1_macro=0.7416  Acc=0.8395

✓ All 9 ANN models trained and saved


## Cell 6 — CNN Architecture and CNN × 9 Feature Sets

Defines `CNN1D`. Inside `forward()`, `x.unsqueeze(1)` reshapes `(B, 15)` to `(B, 1, 15)` for the convolutional layers. After `MaxPool1d(2)` the sequence length reduces from 15 → 7, giving a flattened size of 64 × 7 = 448. The same `train_dl`, `save_dl_plots`, `save_dl_cm`, and `eval_metrics` helpers from Cell 5 are reused — including early stopping on validation macro F1 and F1 vs epoch plots. Results are appended to `all_dl_results`.

In [6]:
class CNN1D(nn.Module):
    """Conv(1→32,k=3)→ReLU→MaxPool(2)→Conv(32→64,k=3)→ReLU→Flatten(448)→Linear(64)→ReLU→Dropout→Linear(3)"""
    def __init__(self, input_len=15, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU()
        )
        flat_size = 64 * (input_len // 2)   # 64 × 7 = 448
        self.clf = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 3)
        )
    def forward(self, x):
        x = x.unsqueeze(1)              # (B, 15) → (B, 1, 15)
        return self.clf(self.conv2(self.conv1(x)))


for method, fd in feature_sets.items():
    print(f'\n── CNN | {method.upper()} ──')
    X_tr_all, y_tr_all = fd['train_X'], fd['train_y']
    X_te,     y_te     = fd['test_X'],  fd['test_y']

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr_all, y_tr_all, test_size=0.2, random_state=SEED, stratify=y_tr_all
    )

    model = CNN1D(input_len=X_tr.shape[1])
    model, history = train_dl(model, X_tr, y_tr, X_val, y_val)

    model.eval()
    with torch.no_grad():
        y_pred = model(torch.FloatTensor(X_te)).argmax(1).numpy()

    metrics = eval_metrics(y_te, y_pred)
    print(f'  Test F1_macro={metrics["F1_Macro"]}  Acc={metrics["Accuracy"]}')

    dl_dir  = os.path.join('models',  'Deep Learning', method)
    fig_dir = os.path.join('figures', 'Deep Learning', method)
    os.makedirs(dl_dir,  exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)

    torch.save(model.state_dict(), os.path.join(dl_dir, 'cnn_model.pt'))
    save_dl_plots(history, 'CNN', method, fig_dir)
    save_dl_cm(y_te, y_pred, 'CNN', method, fig_dir)

    all_dl_results.append({'Feature_Method': method, 'Model': 'CNN', **metrics})

print('\n✓ All 9 CNN models trained and saved')


── CNN | RFE ──
    Early stop at epoch 28  (best val F1_macro=0.9580)
  Test F1_macro=0.8335  Acc=0.8963

── CNN | SKB ──
    Early stop at epoch 27  (best val F1_macro=0.9582)
  Test F1_macro=0.8348  Acc=0.8889

── CNN | FSCS ──
    Early stop at epoch 37  (best val F1_macro=0.9629)
  Test F1_macro=0.8232  Acc=0.8963

── CNN | ETC ──
    Early stop at epoch 45  (best val F1_macro=0.9612)
  Test F1_macro=0.8585  Acc=0.9235

── CNN | PC ──
    Early stop at epoch 40  (best val F1_macro=0.9629)
  Test F1_macro=0.838  Acc=0.8914

── CNN | MI ──
    Early stop at epoch 47  (best val F1_macro=0.9645)
  Test F1_macro=0.8471  Acc=0.9185

── CNN | MIR ──
    Early stop at epoch 26  (best val F1_macro=0.9613)
  Test F1_macro=0.8593  Acc=0.9062

── CNN | MU ──
    Early stop at epoch 26  (best val F1_macro=0.9581)
  Test F1_macro=0.8408  Acc=0.9086

── CNN | VT ──
    Early stop at epoch 23  (best val F1_macro=0.9190)
  Test F1_macro=0.7736  Acc=0.8296

✓ All 9 CNN models trained and saved


## Cell 7 — DL Results Summary CSV

Combines all 18 DL results (9 ANN + 9 CNN) into a DataFrame. Per-method CSVs are saved first (one ANN + one CNN row each), then the full 18-row summary. The table is sorted by **F1_Macro** — consistent with the ML summary. A completion checklist of all outputs is printed.

In [7]:
dl_summary = pd.DataFrame(all_dl_results)

# Per-method DL CSVs
for method in METHODS:
    res_dir = os.path.join('results', 'Deep Learning', method)
    os.makedirs(res_dir, exist_ok=True)
    (dl_summary[dl_summary['Feature_Method'] == method]
     .to_csv(os.path.join(res_dir, 'results_deep_learning.csv'), index=False))

# Overall DL summary
os.makedirs(os.path.join('summary', 'Results', 'Deep Learning'), exist_ok=True)
DL_SUMMARY = os.path.join('summary', 'Results', 'Deep Learning', 'dl_results_summary.csv')
dl_summary.to_csv(DL_SUMMARY, index=False)

print(f'✓ Saved : {DL_SUMMARY}')
print(f'  Shape : {dl_summary.shape}  (expected 18 rows × 7 cols)')
print()
print('Top 5 DL results by F1_Macro (primary metric):')
print(
    dl_summary
    .sort_values('F1_Macro', ascending=False)
    [['Feature_Method', 'Model', 'F1_Macro', 'Accuracy']]
    .head(5)
    .to_string(index=False)
)
print()
print('── Notebook 03 complete ──')
print('  models/Machine Learning/   72 .pkl models  (8 algorithms × 9 methods)')
print('  models/Deep Learning/      18 .pt models   (ANN + CNN × 9 methods)')
print('  figures/Machine Learning/  72 confusion matrix PNGs')
print('  figures/Deep Learning/     36 PNGs  (18 ann/cnn_training.png + 18 confusion)')
print('  results/Machine Learning/   9 per-method CSVs')
print('  results/Deep Learning/      9 per-method CSVs')
print('  summary/Results/           ml_results_summary.csv + dl_results_summary.csv')
print('  Primary metric throughout  : F1_Macro')

✓ Saved : summary\Results\Deep Learning\dl_results_summary.csv
  Shape : (18, 7)  (expected 18 rows × 7 cols)

Top 5 DL results by F1_Macro (primary metric):
Feature_Method Model  F1_Macro  Accuracy
           mir   CNN    0.8593    0.9062
           etc   CNN    0.8585    0.9235
           etc   ANN    0.8577    0.9210
            mu   ANN    0.8484    0.9136
            mi   CNN    0.8471    0.9185

── Notebook 03 complete ──
  models/Machine Learning/   72 .pkl models  (8 algorithms × 9 methods)
  models/Deep Learning/      18 .pt models   (ANN + CNN × 9 methods)
  figures/Machine Learning/  72 confusion matrix PNGs
  figures/Deep Learning/     36 PNGs  (18 ann/cnn_training.png + 18 confusion)
  results/Machine Learning/   9 per-method CSVs
  results/Deep Learning/      9 per-method CSVs
  summary/Results/           ml_results_summary.csv + dl_results_summary.csv
  Primary metric throughout  : F1_Macro
